# 🚀 Automated AI Deployment: Qwen3.8 Abliterated / WhiteRabbitNeo-33B + ComfyUI Studio
**Compatible with:** Google Colab (T4 / L4 / A100 GPU) & Kaggle (T4 GPU)

This high-capacity AI stack provides:
1. **Qwen3.8-27B Unrestricted & Abliterated**: Flagship coding and reasoning LLM with safety refusals removed.
2. **WhiteRabbitNeo-33B**: Specialized cybersecurity, penetration testing, and offensive/defensive lab model.
3. **OpenAI API & Cloudflare Tunnel**: Direct endpoint mapping to Termux (`grok` and `ag-cloud`).
4. **ComfyUI Studio**: SDXL Turbo, ControlNet OpenPose, IP-Adapter FaceID, and AnimateDiff Lightning.


In [ ]:
# ============================================================
# Step 1: Environment Detection, Dependencies & Cloudflare Tunnel
# ============================================================
import os, sys, subprocess, time, re, threading, json

# Detect environment
IS_KAGGLE = os.path.exists("/kaggle")
BASE_DIR = "/kaggle/working" if IS_KAGGLE else "/content"
print(f"[*] Detected Environment: {'Kaggle' if IS_KAGGLE else 'Google Colab'}")
print(f"[*] Base Directory: {BASE_DIR}")

# Verify GPU
gpu_info = subprocess.getoutput("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader")
print(f"[*] Active GPU: {gpu_info}")

# Install cloudflared
print("[*] Installing cloudflared binary...")
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

def run_tunnel(port, name):
    cmd = f"cloudflared tunnel --url http://localhost:{port}"
    proc = subprocess.Popen(cmd.split(), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if match:
            print(f"
" + "="*60)
            print(f"[🚀 PUBLIC URL] {name}: {match.group(0)}")
            print("="*60 + "
")
            break


In [ ]:
# ============================================================
# Step 2: High-Power Unrestricted Models: Qwen3.8-27B & WhiteRabbitNeo-33B
# ============================================================
print('[*] Installing llama-cpp-python & Ollama server with GPU acceleration...')
!pip install -q huggingface_hub
!curl -fsSL https://ollama.com/install.sh | sh

print('[*] Starting Ollama daemon in background...')
os.environ['OLLAMA_HOST'] = '0.0.0.0:11434'
subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)

# Expose via Cloudflare Tunnel
t_ollama = threading.Thread(target=run_tunnel, args=(11434, 'Unrestricted Models OpenAI Endpoint (/v1)'), daemon=True)
t_ollama.start()
time.sleep(4)

# Download High-End Qwen3 & WhiteRabbitNeo GGUF weights directly from Hugging Face
os.makedirs('models/llm', exist_ok=True)
print('[*] Downloading Huihui-Qwen3.8-27B-abliterated (Flagship Coding & Reasoning)...')
from huggingface_hub import hf_hub_download
qwen3_path = hf_hub_download(
    repo_id='huihui-ai/Huihui-Qwen3.8-27B-abliterated-GGUF',
    filename='Qwen3.8-27B-abliterated.Q4_K_M.gguf',
    local_dir='models/llm'
)
print(f'[✓] Qwen3.8-27B Ready: {qwen3_path}')

print('[*] Downloading WhiteRabbitNeo-33B (Elite Pentesting & Cyber Lab Model)...')
wrn_path = hf_hub_download(
    repo_id='manuelgutierrez/WhiteRabbitNeo-33B-v1.5-GGUF',
    filename='WhiteRabbitNeo-33B-v1.5.Q4_K_M.gguf',
    local_dir='models/llm'
)
print(f'[✓] WhiteRabbitNeo-33B Ready: {wrn_path}')

# Create Ollama Modelfiles to serve them via OpenAI API
with open('Modelfile.qwen3', 'w') as f:
    f.write(f'FROM {qwen3_path}\nPARAMETER temperature 0.7\nPARAMETER stop "<|im_end|>"\n')
!ollama create qwen3-27b-abliterated -f Modelfile.qwen3

with open('Modelfile.wrn33b', 'w') as f:
    f.write(f'FROM {wrn_path}\nPARAMETER temperature 0.6\n')
!ollama create whiterabbitneo-33b -f Modelfile.wrn33b

print('\n[✓] Both Qwen3.8-27B-abliterated and WhiteRabbitNeo-33B are live on /v1 OpenAI API!')


In [ ]:
# ============================================================
# Step 3: Setup ComfyUI + SDXL-Turbo, ControlNet (Pose) & IP-Adapter (Face)
# ============================================================
comfy_dir = os.path.join(BASE_DIR, 'ComfyUI')

if not os.path.exists(comfy_dir):
    print(f'[*] Cloning ComfyUI to {comfy_dir}...')
    !git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git {comfy_dir}

%cd {comfy_dir}

print('[*] Installing PyTorch and dependencies...')
!pip install -q torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu121
!pip install -q -r requirements.txt
!pip install -q accelerate transformers insightface onnxruntime-gpu

# Custom Nodes: ComfyUI Manager, AnimateDiff, IPAdapter Plus, ControlNet Auxiliary Preprocessors
nodes = [
    ('ComfyUI-Manager', 'https://github.com/ltdrdata/ComfyUI-Manager.git'),
    ('ComfyUI-AnimateDiff-Evolved', 'https://github.com/Kosinkadink/ComfyUI-AnimateDiff-Evolved.git'),
    ('ComfyUI_IPAdapter_plus', 'https://github.com/cubiq/ComfyUI_IPAdapter_plus.git'),
    ('comfyui_controlnet_aux', 'https://github.com/Fannovel16/comfyui_controlnet_aux.git')
]
for name, git_url in nodes:
    target = os.path.join('custom_nodes', name)
    if not os.path.exists(target):
        print(f'[*] Cloning {name}...')
        !git clone --depth 1 {git_url} {target}

# SDXL-Turbo & SD 1.5 Base
print('[*] Downloading Checkpoints...')
!wget -q -c -P models/checkpoints/ https://huggingface.co/stabilityai/sdxl-turbo/resolve/main/sd_xl_turbo_1.0_fp16.safetensors
!wget -q -c -P models/checkpoints/ https://huggingface.co/runwayml/stable-diffusion-v1-5/resolve/main/v1-5-pruned-emaonly.safetensors

# ControlNet OpenPose (for body posture transfer)
print('[*] Downloading ControlNet OpenPose...')
os.makedirs('models/controlnet', exist_ok=True)
!wget -q -c -P models/controlnet/ https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_v11p_sd15_openpose_fp16.safetensors

# IP-Adapter & FaceID models (for face & character identity transfer)
print('[*] Downloading IP-Adapter Face & Vision Encoders...')
os.makedirs('models/ipadapter', exist_ok=True)
os.makedirs('models/clip_vision', exist_ok=True)
!wget -q -c -P models/ipadapter/ https://huggingface.co/h94/IP-Adapter/resolve/main/models/ip-adapter-plus-face_sd15.safetensors
!wget -q -c -P models/clip_vision/ https://huggingface.co/h94/IP-Adapter/resolve/main/models/image_encoder/model.safetensors

# AnimateDiff Lightning
os.makedirs('models/animatediff_models', exist_ok=True)
!wget -q -c -P models/animatediff_models/ https://huggingface.co/ByteDance/AnimateDiff-Lightning/resolve/main/animatediff_lightning_4step_diffusers.safetensors

print('\n[✓] ComfyUI Studio setup complete with FaceID & OpenPose!')


In [ ]:
# ============================================================
# Step 4: Launch ComfyUI with Cloudflare Tunnel
# ============================================================
# Start Cloudflare Tunnel for ComfyUI (port 8188)
t_comfy = threading.Thread(target=run_tunnel, args=(8188, "ComfyUI Web Interface"), daemon=True)
t_comfy.start()
time.sleep(3)

# Start ComfyUI Server
print("[*] Starting ComfyUI Web Server...")
!python main.py --listen 0.0.0.0 --port 8188 --preview-method auto
